# CNN + BiLSTM + Attention Gesture Classifier

> **COMP3308 context** — This notebook builds the most sophisticated model in the series.
> It combines three ideas you have (partly) already met:
> **CNNs** (week 9), the new **LSTM / BiLSTM** sequence model, and the **attention mechanism**
> which is the core innovation behind modern language models (GPT, BERT, etc.).

---

## Why a three-stage architecture?

Each stage handles a different aspect of the recognition problem.

### Stage 1 — Convolutional Frontend (local feature extraction)

Your COMP3308 lectures covered CNNs on images.  Here we use **1-D convolution** along the
**time axis** of the sensor stream.  Think of each filter as a template detector:

```
Kernel size 7  →  the filter looks at a 7-timestep (~0.23 s) window of all sensor channels simultaneously
```

A filter that fires when it sees "all flex sensors increase sharply together" encodes a
finger-curl event.  Stacking two Conv1D blocks (64 → 128 filters) and halving the time
dimension with MaxPooling after each block compresses the sequence while building
progressively abstract feature representations — exactly as in image CNNs.

**Shape trace through the CNN (example with 156 timesteps, F features):**

```
Input         (156, F)
Conv1D(64)    (156, 64)   ← same padding keeps time dimension
MaxPool(2)    (78,  64)   ← halves timesteps
Conv1D(128)   (78,  128)
MaxPool(2)    (39,  128)  ← halves again
```

### Stage 2 — Bidirectional LSTM (long-range sequential context)

**What is an LSTM?**

A standard LSTM is a recurrent neural network that processes a sequence step by step.
At each timestep t it maintains a **hidden state** h_t — a vector that summarises
everything it has seen so far — and a **cell state** c_t — longer-term memory.

Three sigmoid **gates** (values between 0 and 1) control what to remember and forget:
- **Forget gate** f_t = σ(W_f · [h_{t-1}, x_t] + b_f)  ← how much of c_{t-1} to keep
- **Input gate**  i_t = σ(W_i · [h_{t-1}, x_t] + b_i)  ← how much new info to write
- **Output gate** o_t = σ(W_o · [h_{t-1}, x_t] + b_o)  ← how much of cell to expose

The cell update: c_t = f_t ⊙ c_{t-1}  +  i_t ⊙ tanh(W_c · [h_{t-1}, x_t])

The output:      h_t = o_t ⊙ tanh(c_t)

All W, b parameters are learned by **backpropagation** (backprop through time, BPTT),
exactly the same algorithm you studied in COMP3308 — just unrolled over timesteps.

**Why Bidirectional?**

A standard LSTM reads left-to-right: at timestep t it knows what happened *before* t
but is blind to what comes *after* t.  For offline gesture recognition (we have the
full 5-second trial stored) this is wasteful.

A **Bidirectional LSTM** runs TWO LSTMs in parallel:

```
Forward  LSTM:   x_1 → x_2 → x_3 → … → x_T      produces  h⃗_1, h⃗_2, …, h⃗_T
Backward LSTM:   x_T → x_{T-1} → … → x_1         produces  h⃖_T, h⃖_{T-1}, …, h⃖_1
```

Their outputs are **concatenated** at each timestep:  h_t = [h⃗_t  ‖  h⃖_t]

**Concrete gesture example (Rock gesture — fingers curl, thumb extends):**

| Timestep | Forward LSTM knows          | Backward LSTM knows            |
|----------|-----------------------------|--------------------------------|
| t = 20   | fingers just started curling| the gesture will end as a fist |
| t = 78   | half-curl reached            | thumb extended at the end      |
| t = 140  | fingers fully curled         | only the hold phase remains    |

The backward pass gives every timestep a "preview" of where the gesture is going.
Empirically this improves classification of gestures with ambiguous mid-phases.

**Trade-off:** BiLSTM requires 2× the compute and cannot run in real-time (the backward
pass needs the whole sequence upfront).  For stored 5-second trials this is perfectly fine.

### Stage 3 — Attention Mechanism (temporal focus + interpretability)

Not every timestep is equally informative.  For a Rock gesture, the moment the fingers
lock into the fully-curled position (say t ≈ 80–100) is far more distinctive than the
idle period before the gesture started.

**Attention** learns a scalar weight α_t for each timestep and computes:
```
context = Σ_t  α_t · h_t
```
where the α_t sum to 1 (softmax ensures this).  The model is trained end-to-end, so
the α weights are learned to maximise classification accuracy — the network
**automatically learns where to look**.

Two variants are implemented:

**Additive (Bahdanau) attention**
```
e_t  = V · tanh(W · h_t + b)    ← score network: a small learned MLP
α_t  = softmax(e_t)              ← normalise into probabilities
context = Σ_t  α_t · h_t         ← weighted average of hidden states
```
One set of W, V parameters shared across all timesteps.  Lightweight — good for small datasets.

**Multi-head scaled dot-product attention (Transformer-style)**
```
score = (Q · K^T) / sqrt(d_k)    ← dot product scaled to avoid vanishing gradients
α = softmax(score)
output = α · V
```
Splits features into N_HEADS groups and computes attention independently in each "head".
Different heads can attend to different gesture phases simultaneously.

**Bonus — interpretability:** We can extract the α weights after training and plot them
as a heatmap over time.  High α at a particular timestep means the model was "looking
there" when making its decision.  This is a form of **explainability** that purely
black-box models lack.

---

## Paper reference — A-CBLN (Wu et al., 2023)

Wu et al. *"Data glove-based gesture recognition using CNN-BiLSTM model with attention mechanism"*

Architecture: 2D CNN (1×3 kernel, 16→32 filters) → 2× BiLSTM(8) → Attention → Dense(16) → Softmax  
Result: **95.05 % accuracy** on a 7-class handwashing gesture dataset  
Settings: Adam lr=0.001, batch=64, epochs=50

This notebook uses a higher-capacity variant (larger CNN/BiLSTM) suited for the
higher-dimensional dual-hand glove sensor stream.

---

## Data convention

Column naming: `{hand}_{segment}_{loc}_{channel}`  
Example: `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`

Quaternion columns (`qw`, `qx`, `qy`, `qz`) are always excluded.


---
## Cell 1 — Install Dependencies

Standard data-science and deep-learning packages.  `pydot` + `graphviz` are optional —
they let Keras draw a model architecture diagram, but the notebook runs fine without them.


In [1]:
"""
Cell 1 — Install all required packages.

Why pip install inside the notebook?
    Some Jupyter environments (Google Colab, fresh virtual envs) don't pre-install
    TensorFlow / scikit-learn.  Installing programmatically here makes the notebook
    self-contained — running it from top-to-bottom on a clean machine should just work.

Packages needed and why:
    tensorflow    — the deep-learning framework (Keras lives inside tf.keras)
    scikit-learn  — classification_report, confusion_matrix, train/test split
    scipy         — signal processing (Butterworth filter, interpolation)
    pandas        — loading CSV files into DataFrames
    numpy         — numerical arrays (everything in ML is a numpy array)
    matplotlib    — plotting training curves, attention heatmaps
    seaborn       — prettier confusion-matrix heatmaps
"""
import subprocess, sys

packages = [
    "tensorflow",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# Optional: graphviz for plot_model — draws a visual diagram of the network layers
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydot"], check=False)
subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=False)

print("Dependencies ready.")


Dependencies ready.


E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


---
## Cell 2 — Configuration

**All** hyperparameters and data-selection flags live here.  No other cell needs manual
editing.  This "single source of truth" design means you can re-run the whole notebook
with a different sensor set or model size by changing just this cell.


In [2]:
"""
Cell 2 — Centralised configuration.

Why keep all settings in one place?
    Deep learning experiments involve many interdependent choices (learning rate,
    architecture width, which sensors to use, etc.).  Scattering magic numbers
    throughout the code makes experiments hard to reproduce and compare.

    Keeping everything here means:
        1.  You can see at a glance what configuration produced a given result.
        2.  Changing one flag (e.g. USE_LEFT_HAND = False) propagates everywhere.
        3.  Downstream cells reference these names, so they never need editing.

Each setting is explained in detail below.
"""

# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../ML/DynamicTrainingData/TwoSec'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 20.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 40        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az — linear acceleration (detects movement)
USE_ORIENTATION   = True  # yaw, pitch, roll — Euler angles (detects finger angle)

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'butterworth_lp'

# Normalisation applied after resampling.
# Why normalise?  Raw sensor values have very different scales (e.g. flex sensor
# readings might range 0–1023 while accelerometer readings are ±2 g).  Normalising
# prevents large-magnitude features from dominating gradient updates.
#
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)
                          # False = fit scaler on train set, apply to val/test

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
# Three-way split (COMP3308 week 6 — Evaluation).
# Train: update weights.  Val: tune hyperparameters / early stopping.
# Test: final unbiased estimate of generalisation performance.
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────

# CNN frontend — extracts local temporal patterns from the sensor stream
CNN_FILTERS     = [64, 128]   # First conv block learns 64 feature maps,
                               # second learns 128 (increasing abstraction)
CNN_KERNEL_SIZE = 7            # Each filter spans 7 consecutive timesteps (~0.23 s)
CNN_POOL_SIZE   = 2            # MaxPool halves the time dimension after each block
CNN_DROPOUT     = 0.2          # 20% of CNN feature maps randomly zeroed during training
                               # (regularisation — reduces overfitting)

# Bidirectional LSTM — models long-range sequential dependencies
BILSTM_UNITS       = [64, 32]  # Two BiLSTM layers. Each unit contributes h⃗ and h⃖,
                                # so the actual hidden-state width is 2× these values.
BILSTM_DROPOUT     = 0.3       # Dropout on the LSTM outputs between timesteps
BILSTM_REC_DROPOUT = 0.2       # Dropout on the recurrent connections (h_{t-1} → h_t)
                                # Recurrent dropout specifically combats overfitting
                                # in the sequential path.

# Attention mechanism
# 'additive'    — Bahdanau-style (fewer parameters, good for small datasets)
# 'scaled_dot'  — Multi-head Transformer-style (more expressive, needs more data)
ATTENTION_TYPE  = 'additive'
ATTENTION_HEADS = 4            # Only used when ATTENTION_TYPE = 'scaled_dot'
                               # 4 heads: each attends to different gesture aspects

# Dense classification head — maps the context vector to class probabilities
DENSE_UNITS  = [64]            # One hidden layer of 64 units before softmax
DROPOUT_RATE = 0.3             # Regularisation on the dense layer

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80       # Maximum passes through the training data
BATCH_SIZE          = 32       # Number of samples per gradient update step
                               # (mini-batch gradient descent — COMP3308 week 4)
LEARNING_RATE       = 1e-3     # Adam step size (1e-3 is Adam's widely-used default)
EARLY_STOP_PATIENCE = 15       # Stop if val accuracy doesn't improve for 15 epochs


---
## Cell 3 — Build Feature Column List from Config

The `build_column_groups` utility converts the boolean flags set above into the exact
list of CSV column names to use as model features.  This keeps all sensor-topology
knowledge in one place (`data_loader.py`) rather than scattered across notebooks.

The output `feature_cols` is a Python list of strings like
`['left_thumb_mid_ax', 'left_thumb_mid_ay', ...]` — one string per sensor channel.


In [3]:
"""
Cell 3 — Construct the canonical list of sensor feature columns.

What this cell does:
    1. Adds the shared utilities folder to Python's module search path.
    2. Imports the sensor topology constants (HANDS, FINGERS, LOCS, etc.) and
       the column-builder function from utils/data_loader.py.
    3. Uses the CONFIG flags from Cell 2 to decide which hands/locations/channels
       to include.
    4. Calls build_column_groups() to produce the final column-name list.
    5. Prints a grouped summary so you can verify the selection looks correct.

Why a helper function instead of a hard-coded list?
    The glove has 191 raw columns.  Selecting subsets (e.g. left hand only,
    or orientation channels only) manually would require editing a long list
    every time you change a flag.  build_column_groups() does this automatically
    from the sensor topology constants, so toggling USE_LEFT_HAND in Cell 2 is
    the only change needed.
"""
import sys, os
# Add the parent directory to the Python path so we can import from utils/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from utils.data_loader import (
    build_column_groups,   # builds the feature column list from topology constants
    HANDS,                 # ["right", "left"]
    FINGERS,               # ["thumb", "index", "middle", "ring", "pinky"]
    LOCS,                  # ["mid", "prox"] — distal and proximal phalanx IMUs
    IMU_ALL_CHANNELS,      # ["yaw", "pitch", "roll", "ax", "ay", "az"]
    FLEX_CHANNELS,         # ["mcp_flex", "pip_flex"]
    WRIST_ALL_CHANNELS,    # ["ax", "ay", "az", "heading", "pitch", "roll"]
)

# ── Resolve CONFIG flags into filtered lists ──────────────────────────────────
# Only include hands that have their flag set to True
_hands = (
    [h for h in HANDS if (USE_LEFT_HAND  and h == "left")
                      or (USE_RIGHT_HAND and h == "right")]
    or HANDS  # fallback: both hands if both flags are False (shouldn't happen)
)

_fingers = FINGERS  # All five fingers are always included

# Only include the IMU locations (distal / proximal) that are enabled
_locs = [
    l for l in LOCS
    if (USE_DISTAL_IMU   and l == "mid")   # distal phalanx (fingertip segment)
    or (USE_PROXIMAL_IMU and l == "prox")  # proximal phalanx (base segment)
] or LOCS  # fallback to all locations

# ── Call build_column_groups to get the final feature list ────────────────────
feature_cols = build_column_groups(
    hands               = _hands,
    fingers             = _fingers,
    locs                = _locs,
    include_flex        = USE_FLEX_SENSORS,   # MCP and PIP joint flex sensors
    include_palm_prox   = USE_PALM_IMU,        # back-of-hand IMU
    include_wrist       = USE_WRIST_IMU,       # wrist IMU
    include_accel       = USE_ACCELEROMETER,   # ax, ay, az channels
    include_orient      = USE_ORIENTATION,     # yaw/pitch/roll channels
)

print(f"Feature columns selected: {len(feature_cols)}")
print()

# ── Print a grouped summary for inspection ────────────────────────────────────
# Group columns by their sensor prefix (e.g. 'left_thumb_mid') for readability
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)

for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


ImportError: cannot import name 'filter_existing_columns' from 'utils.data_loader' (/home/jestin/ThesisRepo/ML/utils/data_loader.py)

---
## Cell 4 — Imports and Load Dataset

This cell imports the full scientific Python and TensorFlow stack, then calls
`build_dataset` to walk the data directory, load every gesture trial CSV, apply
preprocessing, and return a 3-D NumPy array `X` of shape `(N, T, C)` plus an
integer label array `y` of shape `(N,)`.

The resulting `X` is exactly what Keras expects as input to the first layer.


In [ ]:
"""
Cell 4 — Imports and dataset loading.

Imports are grouped by purpose so you can see at a glance what each package is for.

DATA REPRESENTATION (reminder from COMP3308):
    After this cell, X has shape (N, T, C) where:
        N = number of gesture trials (e.g. 400 — 4 classes × 100 trials each)
        T = TARGET_LEN timesteps per trial (e.g. 156)
        C = number of sensor channels (e.g. 120 with all sensors enabled)

    Each X[i] is a 2-D matrix representing one 5-second gesture recording.
    y[i] is the integer class label (0=Fist, 1=Flat, 2=Okay, 3=Rock).

    Think of it as: N training examples, each with a (T × C) feature matrix.
    Your COMP3308 perceptron had a 1-D feature vector per example; here each
    example is a 2-D time-series matrix — but the supervised learning setup
    (labelled examples, train/val/test split, accuracy metric) is identical.
"""

# ── Standard library ──────────────────────────────────────────────────────────
import os           # file-path manipulation (os.path.join, os.makedirs)
import json         # save results as a JSON file at the end

# ── Numerical computing ───────────────────────────────────────────────────────
import numpy as np           # everything in ML is ultimately a numpy array

# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd          # reading CSV files, building DataFrames

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt    # line plots, bar charts
import seaborn as sns              # heatmap for confusion matrix

# ── Machine learning utilities ────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,   # per-class precision, recall, F1
    confusion_matrix,        # NxN confusion matrix
)

# ── Deep learning — TensorFlow / Keras ───────────────────────────────────────
import tensorflow as tf
import tensorflow.keras as keras

# Keras building blocks (Functional API)
from tensorflow.keras.layers import (
    Input,                  # defines the entry point / shape
    Conv1D,                 # 1-D convolution along the time axis
    BatchNormalization,     # normalises layer activations (stabilises training)
    MaxPooling1D,           # reduces temporal resolution (extracts dominant features)
    Dropout,                # randomly zeros units during training (prevents overfitting)
    Bidirectional,          # wrapper that runs an RNN in both directions
    LSTM,                   # Long Short-Term Memory cell
    Dense,                  # fully connected layer
    LayerNormalization,     # layer-level normalisation (used in Transformer blocks)
    GlobalAveragePooling1D, # averages a sequence over time → single vector
    MultiHeadAttention,     # Transformer-style multi-head attention
)
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import plot_model

# Keras training callbacks
from tensorflow.keras.callbacks import (
    EarlyStopping,      # stops training when val metric stops improving
    ReduceLROnPlateau,  # reduces learning rate when training plateaus
    ModelCheckpoint,    # saves the best model weights to disk automatically
)

# ── Project utilities (in utils/data_loader.py) ───────────────────────────────
from utils.data_loader import build_dataset, split_dataset

# ── Output directories ────────────────────────────────────────────────────────
RESULTS_DIR = 'results'
MODEL_DIR   = 'saved_models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,   exist_ok=True)

# =============================================================================
# LOAD DATASET
# =============================================================================
# build_dataset() orchestrates the full preprocessing pipeline:
#   1. Discover all CSV files under DATA_ROOT (one file = one 5-second trial)
#   2. Load each CSV and select only the columns in feature_cols
#   3. Apply optional temporal filter (FILTER_TYPE)
#   4. Resample every trial to exactly TARGET_LEN timesteps
#   5. Normalise sensor values (NORMALIZATION)
#   6. Stack all trials into X (N, T, C) and labels into y (N,)
#
# feature_cols was built in Cell 3.
# All CONFIG flags are already set — no changes needed here.

X, y, LABELS, feature_cols = build_dataset(
    data_root      = DATA_ROOT,
    feature_cols   = feature_cols,   # from Cell 3
    filter_type    = FILTER_TYPE,
    fs             = FS_HZ,
    target_len     = TARGET_LEN,
    normalization  = NORMALIZATION,
    per_sample_norm= PER_SAMPLE_NORM,
)

# ── Derive key dimensions ─────────────────────────────────────────────────────
N_SAMPLES  = X.shape[0]   # total number of trials
N_FEATURES = X.shape[2]   # number of sensor channels (width of each timestep)
N_CLASSES  = len(LABELS)  # number of gesture classes

print(f"\nDataset summary:")
print(f"  X shape      : {X.shape}  — (samples, timesteps, features)")
print(f"  y shape      : {y.shape}")
print(f"  Classes ({N_CLASSES}) : {LABELS}")
print(f"  N_FEATURES   : {N_FEATURES}")
print(f"  Dtype        : {X.dtype}")

# ── Sanity check: class balance ───────────────────────────────────────────────
# A balanced dataset (equal samples per class) is important for unbiased training.
# With 400 samples / 4 classes we expect 100 per class.
print("\nClass distribution:")
for i, lbl in enumerate(LABELS):
    count = int((y == i).sum())
    print(f"  {lbl:20s}  {count:4d}  ({100*count/N_SAMPLES:.1f}%)")

# ── Class distribution bar chart ─────────────────────────────────────────────
counts = pd.Series(y).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([LABELS[i] for i in counts.index], counts.values, color='steelblue', edgecolor='white')
ax.set_title('Dataset class distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Gesture class')
ax.set_ylabel('Number of trials')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(v), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 5 — Split into Train / Validation / Test Sets

We divide the 400 labelled trials into three non-overlapping subsets:

| Subset     | Ratio | Purpose                                                      |
|------------|-------|--------------------------------------------------------------|
| Train      | 70 %  | Model sees these samples and updates weights                 |
| Validation | 10 %  | Monitors generalisation; used by early stopping              |
| Test       | 20 %  | Final unbiased accuracy reported in the thesis               |

**Why three sets?** Your COMP3308 lectures (week 6) explain that evaluating on the same
data used for training produces an optimistic (over-fitted) accuracy estimate.  The
validation set lets us tune hyperparameters and stop training early without "peeking"
at the test set.  The test set is only touched once, at the very end.


In [ ]:
"""
Cell 5 — Stratified train / validation / test split.

'Stratified' means the split preserves the class distribution in each subset.
E.g. if Fist = 25% overall, then 25% of train, val, and test will be Fist.
This prevents unlucky splits where one subset has almost no examples of a class.

One-hot encoding:
    Keras 'categorical_crossentropy' loss expects the target as a one-hot vector.
    y_train integer labels (e.g. [0, 2, 1, ...]) are converted to:
        y_train_oh  →  [[1,0,0,0], [0,0,1,0], [0,1,0,0], ...]

    This is equivalent to how your COMP3308 lectures describe multi-class output nodes:
    each output node fires for one class, and the target is a 1 at the correct class index.
"""

# split_dataset() wraps sklearn's StratifiedShuffleSplit for reproducible stratified splits.
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,   # fixing the seed makes results reproducible
)

print("\nSplit summary:")
print(f"  Train : {X_train.shape}  labels {y_train.shape}")
print(f"  Val   : {X_val.shape}  labels {y_val.shape}")
print(f"  Test  : {X_test.shape}  labels {y_test.shape}")

# ── One-hot encode integer labels for categorical crossentropy ────────────────
# Integer class 2 (Okay)  →  one-hot [0, 0, 1, 0]
# Integer class 0 (Fist)  →  one-hot [1, 0, 0, 0]
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)

# ── Per-split class balance bar chart ─────────────────────────────────────────
# Verify the stratified split maintained the class distribution.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ys, title in zip(axes,
                         [y_train, y_val, y_test],
                         ['Train', 'Validation', 'Test']):
    counts_s = pd.Series(ys).value_counts().sort_index()
    ax.bar(
        [LABELS[i] for i in counts_s.index],
        counts_s.values,
        color='steelblue', edgecolor='white',
    )
    ax.set_title(f'{title} set — {len(ys)} samples', fontsize=10, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_split_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 6 — Model Definition: CNN + BiLSTM + Attention

This cell defines the full architecture as a Keras *Functional API* model.

**Why Functional API and not Sequential?**  
The attention layer needs to operate on the *whole sequence* from the BiLSTM and then
produce a single context vector — this is a non-linear topology (branch and merge) that
`Sequential` cannot express.  The Functional API lets us connect layers like a graph.

**Shape trace (with default CONFIG — 156 timesteps, F features):**

```
Input layer        (batch, 156, F)
─── CNN Block 1 ───
  Conv1D(64, k=7)  (batch, 156, 64)   ← 'same' padding keeps T unchanged
  BatchNorm        (batch, 156, 64)
  MaxPool(2)       (batch,  78, 64)   ← T halved
  Dropout(0.2)     (batch,  78, 64)
─── CNN Block 2 ───
  Conv1D(128, k=7) (batch,  78, 128)
  BatchNorm        (batch,  78, 128)
  MaxPool(2)       (batch,  39, 128)  ← T halved again
  Dropout(0.2)     (batch,  39, 128)
─── BiLSTM Stack ───
  BiLSTM(64)       (batch,  39, 128)  ← 2×64 units (concat of forward+backward)
  BatchNorm        (batch,  39, 128)
  BiLSTM(32)       (batch,  39,  64)  ← 2×32 units
  BatchNorm        (batch,  39,  64)
─── Attention ───
  Additive attn    context: (batch, 64)   weights α: (batch, 39)
─── Dense Head ───
  Dense(64, ReLU)  (batch, 64)
  Dropout(0.3)     (batch, 64)
  Dense(4, softmax)(batch, 4)   ← class probabilities (one per gesture)
```

The model is compiled to train only the primary `output` head; `attn_weights` is a
pass-through output for interpretability purposes.


In [ ]:
"""
Cell 6 — CNN + BiLSTM + Attention model.

ARCHITECTURE OVERVIEW
─────────────────────
The three stages handle distinct aspects of the sequence classification problem:

Stage 1 — CNN Frontend
    1D convolution slides a kernel across the time axis.
    Each filter learns a local pattern detector (e.g. "sharp flex increase over 0.2 s").
    MaxPooling reduces the temporal resolution, acting as a coarse-graining step.
    BatchNormalization speeds up training by keeping activations in a stable range.
    After two blocks, the sequence is shorter (less time) but deeper (more features).

Stage 2 — Bidirectional LSTM
    The shortened but feature-rich sequence enters a stack of BiLSTM layers.
    Each BiLSTM layer returns a FULL output sequence (return_sequences=True) because
    the attention layer in Stage 3 needs per-timestep hidden states, not just h_T.

    Forward LSTM:  processes x_1 → … → x_T  producing h⃗_1, …, h⃗_T
    Backward LSTM: processes x_T → … → x_1  producing h⃖_T, …, h⃖_1
    merge_mode='concat': h_t = [h⃗_t, h⃖_t]  (output width = 2 × LSTM units)

Stage 3 — Attention
    Takes the BiLSTM sequence (batch, T, H) and collapses it to (batch, H) by
    computing a weighted average where the weights are learned.

    Additive (Bahdanau):
        score_t = V · tanh(W · h_t)    — each timestep scored by a small MLP
        α_t = softmax(scores)           — normalised to sum to 1
        context = Σ_t  α_t · h_t       — weighted average → context vector

    Multi-head (scaled dot-product):
        Uses tf.keras.layers.MultiHeadAttention — Transformer convention.
        score = Q · K^T / sqrt(d_k)     — scaled dot product (avoids small gradients)
        The residual connection (attn_out + bilstm_out) and LayerNorm follow
        the standard Transformer block design (Vaswani et al., 2017).

Attention weights are EXPOSED as a second model output so we can visualise them later.
"""

# ── Custom Additive (Bahdanau) Attention Layer ────────────────────────────────
class BahdanauAttention(layers.Layer):
    """
    Additive (Bahdanau-style) attention over a temporal sequence.

    Reference: Bahdanau, D., Cho, K., Bengio, Y. (2015).
               "Neural Machine Translation by Jointly Learning to Align and Translate."
               ICLR 2015.  (This is where the attention idea was first popularised.)

    How it works:
        Given a sequence of hidden states h_1, …, h_T (each a vector of size H):

        Step 1 — Score each timestep with a tiny learned network:
            e_t = V · tanh(W · h_t + b)
            W is (units × H), V is (1 × units).  This is the "energy" for timestep t.

        Step 2 — Normalise scores to a probability distribution:
            α_t = softmax(e_t)       α_t ∈ [0, 1],  Σ_t α_t = 1

        Step 3 — Compute the context vector as a weighted sum:
            c = Σ_t  α_t · h_t       shape: (H,)

        The network learns W and V during backpropagation so that the α weights
        focus on whichever timesteps are most useful for classification.

    Args:
        units : width of the intermediate scoring MLP (W matrix width)

    Input shape  : (batch, T, H)   — BiLSTM output sequence
    Output shapes: context (batch, H), alpha (batch, T)
    """

    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        # W : projects each hidden state h_t to an intermediate space of size `units`
        self.W = layers.Dense(units, use_bias=True)
        # V : maps the intermediate representation to a scalar score
        self.V = layers.Dense(1, use_bias=False)

    def call(self, hidden_seq, training=False):
        """
        Forward pass.

        Args:
            hidden_seq : tensor of shape (batch, T, H)
                         The full BiLSTM output sequence — one H-vector per timestep.
            training   : bool passed by Keras (used by Dropout/BN sub-layers if any)

        Returns:
            context    : (batch, H)  — the weighted sum, ready for the Dense head
            alpha      : (batch, T)  — the attention weights (for visualisation)
        """
        # ── Step 1: Score each timestep ───────────────────────────────────────
        # self.W(hidden_seq) : (batch, T, H) → (batch, T, units)
        # tf.nn.tanh(...)    : element-wise non-linearity (squashes to [-1, 1])
        # self.V(...)        : (batch, T, units) → (batch, T, 1)  (scalar per timestep)
        score = self.V(tf.nn.tanh(self.W(hidden_seq)))  # (batch, T, 1)

        # ── Step 2: Normalise scores to probabilities (softmax over T) ────────
        # axis=1 ensures softmax is computed along the time axis, not features.
        # After softmax, alpha[b, t, 0] is the attention weight for sample b, time t.
        alpha = tf.nn.softmax(score, axis=1)            # (batch, T, 1)

        # ── Step 3: Weighted sum of hidden states (the context vector) ────────
        # alpha * hidden_seq : broadcasts (batch, T, 1) × (batch, T, H) → (batch, T, H)
        # tf.reduce_sum(..., axis=1) : sum over time → (batch, H)
        context = tf.reduce_sum(alpha * hidden_seq, axis=1)  # (batch, H)

        # Squeeze the trailing size-1 dimension from alpha for clean output shape
        return context, tf.squeeze(alpha, axis=-1)           # (batch, H), (batch, T)

    def get_config(self):
        """Required for model serialisation (model.save / model.load_model)."""
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


# ── Model builder ─────────────────────────────────────────────────────────────
def build_cnn_bilstm_attention(
        seq_len         = TARGET_LEN,
        n_features      = N_FEATURES,
        n_classes       = N_CLASSES,
        cnn_filters     = CNN_FILTERS,
        cnn_kernel      = CNN_KERNEL_SIZE,
        cnn_pool        = CNN_POOL_SIZE,
        cnn_dropout     = CNN_DROPOUT,
        bilstm_units    = BILSTM_UNITS,
        bilstm_drop     = BILSTM_DROPOUT,
        bilstm_rec_drop = BILSTM_REC_DROPOUT,
        attention_type  = ATTENTION_TYPE,
        attention_heads = ATTENTION_HEADS,
        dense_units     = DENSE_UNITS,
        dropout_rate    = DROPOUT_RATE,
        learning_rate   = LEARNING_RATE,
):
    """
    Build and compile the CNN + BiLSTM + Attention model.

    Uses the Keras Functional API (connect layers explicitly as a graph) rather
    than Sequential (stack layers in a line) because:
        - The attention layer exposes a SECOND output (α weights) for visualisation.
        - Having multiple outputs requires a graph topology.

    Args:
        seq_len         : number of timesteps after CNN pooling  (= TARGET_LEN)
        n_features      : number of sensor channels              (= N_FEATURES)
        n_classes       : number of gesture classes              (= N_CLASSES)
        ... (all other args are the CONFIG hyperparameters)

    Returns:
        Compiled tf.keras.Model with two outputs:
            output       : (batch, n_classes)  — softmax class probabilities
            attn_weights : (batch, T_attn)     — attention weights per timestep
    """

    # ── 1. Input layer ─────────────────────────────────────────────────────────
    # Declares the input shape to Keras. No computation here — just bookkeeping.
    # Shape: (batch, seq_len, n_features) where batch dimension is implicit (None).
    inp = Input(shape=(seq_len, n_features), name='sensor_input')
    # After this: x.shape = (None, 156, N_FEATURES)

    # ── 2. CNN frontend — local temporal feature extraction ────────────────────
    x = inp
    for i, filters in enumerate(cnn_filters):
        # Conv1D: slides `filters` kernels of width `cnn_kernel` along the time axis.
        # padding='same' adds zeros at the edges so the output length equals input length.
        # activation='relu' applies ReLU: max(0, x) — kills negative activations.
        # After Conv1D: (batch, T, filters)
        x = Conv1D(
            filters     = filters,
            kernel_size = cnn_kernel,
            padding     = 'same',       # keeps time dimension unchanged
            activation  = 'relu',
            name        = f'conv1d_{i+1}',
        )(x)

        # BatchNorm: normalises each mini-batch's activations to mean≈0, std≈1.
        # This prevents the "internal covariate shift" problem and allows higher LRs.
        x = BatchNormalization(name=f'bn_conv_{i+1}')(x)

        # MaxPool: takes the maximum value in each non-overlapping window of size cnn_pool.
        # Halves the time dimension: 156 → 78 → 39 (with pool_size=2, two blocks).
        # Why max? The presence of a feature (max activation) is more stable than its
        # exact position — MaxPool provides translation invariance within the window.
        x = MaxPooling1D(pool_size=cnn_pool, name=f'pool_{i+1}')(x)

        # Dropout: randomly zeros cnn_dropout fraction of feature maps during training.
        # At inference time all units are active (scaled by 1-p to compensate).
        # This is a regularisation technique — prevents the model from over-relying on
        # any single feature detector. (COMP3308 week 7: overfitting & regularisation)
        x = Dropout(cnn_dropout, name=f'drop_conv_{i+1}')(x)

    # After CNN stack: shape = (batch, seq_len // pool^n_blocks, cnn_filters[-1])
    # Default: (batch, 39, 128)
    cnn_out = x

    # ── 3. Bidirectional LSTM stack ────────────────────────────────────────────
    for j, units in enumerate(bilstm_units):
        # return_sequences=True is ESSENTIAL here because:
        #   - The attention layer needs h_t at EVERY timestep t (not just h_T).
        #   - If return_sequences=False we'd only get h_T — the final hidden state —
        #     and there would be nothing for the attention layer to weight.
        #
        # The Bidirectional wrapper runs the LSTM twice:
        #   forward  LSTM: t = 1, 2, …, T  →  h⃗_1, h⃗_2, …, h⃗_T
        #   backward LSTM: t = T, T-1, …, 1 →  h⃖_T, h⃖_{T-1}, …, h⃖_1
        #
        # merge_mode='concat' produces h_t = [h⃗_t ‖ h⃖_t], doubling the width.
        # Output shape: (batch, T, 2 * units)
        x = Bidirectional(
            LSTM(
                units,
                return_sequences  = True,         # must be True for attention
                dropout           = bilstm_drop,   # applied to LSTM output
                recurrent_dropout = bilstm_rec_drop, # applied to h_{t-1} → h_t path
                                                      # helps prevent recurrent overfitting
            ),
            merge_mode = 'concat',   # concatenate forward and backward hidden states
            name       = f'bilstm_{j+1}',
        )(x)

        # BatchNorm after each BiLSTM layer for training stability
        x = BatchNormalization(name=f'bn_lstm_{j+1}')(x)

    bilstm_out = x  # shape: (batch, T_cnn_out, 2 * bilstm_units[-1])
                    # Default: (batch, 39, 64)

    # ── 4. Attention layer ─────────────────────────────────────────────────────
    attention_type_lower = attention_type.lower()

    if attention_type_lower == 'additive':
        # ── Bahdanau (additive) attention ─────────────────────────────────────
        # See BahdanauAttention docstring above for full mathematical details.
        # `units` set to match the BiLSTM output width (2 * bilstm_units[-1]).
        attn_layer = BahdanauAttention(
            units = bilstm_units[-1] * 2,   # intermediate MLP width = H
            name  = 'bahdanau_attention',
        )
        # context  : (batch, 2 * bilstm_units[-1])  ← compressed gesture representation
        # attn_weights : (batch, T_cnn_out)           ← one scalar per timestep
        context, attn_weights = attn_layer(bilstm_out)

    elif attention_type_lower == 'scaled_dot':
        # ── Multi-head scaled dot-product attention (Transformer-style) ───────
        # From: Vaswani et al. (2017) "Attention Is All You Need." NeurIPS.
        #
        # In self-attention, query = key = value = the BiLSTM sequence itself.
        # The model learns to attend from each timestep to all other timesteps.
        #
        # key_dim = total feature dim / num_heads  (each head attends over a subspace)
        # Scaling by sqrt(d_k) prevents vanishing gradients from dot products being
        # too large when d_k is high.
        mha = MultiHeadAttention(
            num_heads = attention_heads,
            key_dim   = bilstm_units[-1] * 2 // attention_heads,  # d_k per head
            name      = 'multihead_attention',
        )
        # Query, Key, Value are all the BiLSTM output sequence (self-attention)
        # return_attention_scores=True exposes the α matrix for visualisation
        # attn_scores shape: (batch, num_heads, T_q, T_k)
        attn_out, attn_scores = mha(
            query                   = bilstm_out,
            value                   = bilstm_out,
            key                     = bilstm_out,
            return_attention_scores = True,
        )

        # Residual connection + LayerNorm (standard Transformer convention).
        # Adding bilstm_out back (the residual) lets gradients flow directly through
        # without passing through the attention computation — helps deeper networks train.
        attn_out = LayerNormalization(name='attn_layer_norm')(attn_out + bilstm_out)

        # GlobalAveragePooling over time: average h_t across all T timesteps → (batch, H)
        # This is simpler than additive attention's weighted sum but still pools the sequence.
        context = GlobalAveragePooling1D(name='global_avg_pool')(attn_out)

        # Average attention scores across heads and query positions for 1-D visualisation.
        # attn_scores: (batch, heads, T_q, T_k) → mean over heads and query dim
        #   → (batch, T_k) : a single weight per key timestep per sample
        attn_weights = tf.reduce_mean(attn_scores, axis=[1, 2])  # (batch, T_k)

    else:
        raise ValueError(
            f"Unknown attention_type: '{attention_type}'. "
            "Choose 'additive' or 'scaled_dot'."
        )

    # After attention: context.shape = (batch, 2 * bilstm_units[-1])
    # Default: (batch, 64)

    # ── 5. Dense classification head ───────────────────────────────────────────
    # Maps the context vector (the attention-weighted gesture representation) to
    # class probabilities through a standard fully-connected network.
    x = context
    for k, units in enumerate(dense_units):
        # Dense with ReLU: a standard non-linear transformation layer.
        # ReLU(x) = max(0, x) — efficient and widely used. (COMP3308 week 3)
        x = Dense(units, activation='relu', name=f'dense_{k+1}')(x)
        # Dropout for regularisation in the dense head
        x = Dropout(dropout_rate, name=f'drop_dense_{k+1}')(x)

    # Final softmax layer: maps logits to a valid probability distribution over classes.
    # output[i] = probability that this gesture belongs to class i.
    # Σ output[i] = 1  (enforced by softmax).
    # Loss function: categorical_crossentropy = -Σ_c  y_true_c · log(y_pred_c)
    # (This is the multi-class generalisation of binary cross-entropy — COMP3308 week 5)
    output = Dense(n_classes, activation='softmax', name='output')(x)

    # ── 6. Assemble the model (Functional API) ─────────────────────────────────
    # We expose BOTH outputs:
    #   'output'       — the class probabilities (used during training)
    #   'attn_weights' — the attention coefficients (used for visualisation only)
    # During training Keras only computes loss on 'output'; attn_weights is a
    # free pass-through that doesn't affect gradients.
    model = Model(
        inputs  = inp,
        outputs = [output, attn_weights],
        name    = f'CNN_BiLSTM_{attention_type.capitalize()}Attention',
    )

    # Compile: specifies the optimiser, loss, and evaluation metrics.
    # Adam = Adaptive Moment Estimation: maintains per-parameter learning rates
    #        adjusted by running estimates of gradient mean and variance.
    #        It is the default choice for most modern deep-learning tasks.
    # Loss dictionary targets only the primary output — attn_weights has no loss.
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = {'output': 'categorical_crossentropy'},
        metrics   = {'output': ['accuracy']},
    )
    return model


# ── Instantiate and summarise ─────────────────────────────────────────────────
model = build_cnn_bilstm_attention()

# model.summary() prints the layer stack, output shapes, and parameter counts.
# Look for the shape trace: (None, 156, F) → … → (None, 4)
model.summary()

# Architecture diagram (requires graphviz — skipped gracefully if not installed)
try:
    diagram_path = os.path.join(RESULTS_DIR, '04_model_architecture.png')
    plot_model(
        model,
        to_file          = diagram_path,
        show_shapes      = True,
        show_dtype       = False,
        show_layer_names = True,
        expand_nested    = True,
        dpi              = 96,
    )
    from IPython.display import Image, display
    display(Image(diagram_path))
    print(f"Model diagram saved to {diagram_path}")
except Exception as e:
    print(f"[plot_model skipped] — graphviz not installed: {e}")

# Parameter count
total_params     = model.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


---
## Cell 7 — Training and Learning Curves

`model.fit` runs mini-batch gradient descent for up to `EPOCHS` epochs.
Three **callbacks** automate good training practice:

| Callback             | Purpose                                                         |
|---------------------|-----------------------------------------------------------------|
| `EarlyStopping`     | Stops training when val accuracy stops improving; restores best weights — avoids overfitting by training too long |
| `ReduceLROnPlateau` | Halves the learning rate when val loss plateaus — useful escape from saddle points |
| `ModelCheckpoint`   | Saves the best-seen model to disk so it survives a kernel crash  |

After training, accuracy and loss curves are plotted.  **The gap between training and
validation curves shows the degree of overfitting** — a concept from COMP3308 week 7.


In [ ]:
"""
Cell 7 — Training the model.

TRAINING LOOP RECAP (COMP3308 week 4)
---------------------------------------
Each epoch:
    1. Shuffle training set (ensures mini-batches are random)
    2. For each mini-batch of BATCH_SIZE samples:
        a. Forward pass: compute predictions y_pred
        b. Compute loss: categorical_crossentropy(y_true, y_pred)
        c. Backward pass (backpropagation): compute ∂loss/∂W for all weights
        d. Adam optimiser step: update every weight W ← W - lr * gradient_estimate
    3. Evaluate on validation set (no weight updates)
    4. Check callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)

MINI-BATCH GRADIENT DESCENT
    BATCH_SIZE = 32 means 280 / 32 ≈ 9 weight updates per epoch (on 280 train samples).
    Each update uses a noisy gradient estimate from 32 random samples.
    This noise actually helps escape local minima — it's called stochastic gradient descent.

LEARNING CURVES
    If training accuracy >> validation accuracy: the model is overfitting.
    If both are low: the model is underfitting (not enough capacity or epochs).
    The ideal curve shows both converging to a high value without a large gap.
"""

checkpoint_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attn_best.keras')

callbacks = [
    # EarlyStopping: watch val_output_accuracy (the accuracy on the validation set)
    # patience=15: allow 15 epochs without improvement before stopping
    # restore_best_weights=True: at the end, reload the weights from the epoch with
    #                            the highest val accuracy (not the final epoch's weights)
    EarlyStopping(
        monitor              = 'val_output_accuracy',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,     # revert to best weights after early stop
        verbose              = 1,
    ),
    # ReduceLROnPlateau: if val_output_loss doesn't decrease for 5 epochs,
    # multiply learning rate by 0.5 (halve it).  A lower LR can help fine-tune
    # when the model is close to a good minimum but overshooting.
    ReduceLROnPlateau(
        monitor  = 'val_output_loss',
        factor   = 0.5,        # new_lr = old_lr × 0.5
        patience = 5,          # epochs without improvement before reducing
        min_lr   = 1e-6,       # floor: never reduce below this value
        verbose  = 1,
    ),
    # ModelCheckpoint: saves the full model (architecture + weights + optimiser state)
    # whenever val_output_accuracy improves.  This creates a safety copy on disk.
    ModelCheckpoint(
        filepath          = checkpoint_path,
        monitor           = 'val_output_accuracy',
        save_best_only    = True,    # only overwrite if this epoch is the best so far
        save_weights_only = False,   # save the full model (not just weights)
        verbose           = 0,
    ),
]

print(f"Training CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention model")
print(f"  Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print("─" * 60)

# model.fit returns a History object whose .history attribute is a dict:
# {'output_accuracy': [...], 'val_output_accuracy': [...], 'output_loss': [...], ...}
# Each list has one value per epoch — these are the raw data for the learning curves.
history = model.fit(
    X_train,
    {'output': y_train_oh},              # target dict — only 'output' has a loss
    validation_data = (X_val, {'output': y_val_oh}),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,                 # print one line per epoch
)

# ── Learning curves ───────────────────────────────────────────────────────────
# Determine the correct metric key names (they depend on whether the model has
# named outputs or not — Keras prefixes them with the output name).
hist        = history.history
acc_key     = 'output_accuracy'     if 'output_accuracy'     in hist else 'accuracy'
val_acc_key = 'val_output_accuracy' if 'val_output_accuracy' in hist else 'val_accuracy'
loss_key    = 'output_loss'         if 'output_loss'         in hist else 'loss'
val_loss_key= 'val_output_loss'     if 'val_output_loss'     in hist else 'val_loss'

epochs_ran = range(1, len(hist[loss_key]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(epochs_ran, hist[acc_key],     label='Train accuracy', linewidth=2)
ax1.plot(epochs_ran, hist[val_acc_key], label='Val accuracy',   linewidth=2, linestyle='--')
best_val_acc = max(hist[val_acc_key])
best_epoch   = int(np.argmax(hist[val_acc_key])) + 1
ax1.axvline(
    best_epoch, color='red', linestyle=':', alpha=0.7,
    label=f'Best epoch ({best_epoch})',
)
ax1.set_title(f'Accuracy  |  Best val: {best_val_acc:.4f}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot — loss should decrease; a diverging val_loss is the clearest sign of overfitting
ax2.plot(epochs_ran, hist[loss_key],     label='Train loss', linewidth=2)
ax2.plot(epochs_ran, hist[val_loss_key], label='Val loss',   linewidth=2, linestyle='--')
ax2.axvline(
    best_epoch, color='red', linestyle=':', alpha=0.7,
    label=f'Best epoch ({best_epoch})',
)
ax2.set_title('Cross-Entropy Loss', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(
    f'CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention — Training Curves',
    fontsize=13, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f"\nBest validation accuracy : {best_val_acc:.4f}  at epoch {best_epoch}")
print(f"Model checkpoint saved   : {checkpoint_path}")


---
## Cell 8 — Evaluation on the Test Set

The **test set** has been held out completely until now.  We evaluate here once.

**Metrics (COMP3308 week 6):**

- **Accuracy** = correct predictions / total = easy to understand but misleading on imbalanced data
- **Precision** = TP / (TP + FP) — out of all predicted positives, how many are correct?
- **Recall** = TP / (TP + FN) — out of all actual positives, how many did we find?
- **F1-score** = 2 × (Precision × Recall) / (Precision + Recall) — harmonic mean
- **Confusion matrix** — rows = true classes, columns = predicted classes; the diagonal
  shows correct predictions and off-diagonal entries show specific confusions

Because our dataset is balanced (100 samples per class), accuracy and macro-F1 should
be very similar.


In [ ]:
"""
Cell 8 — Test-set evaluation with classification report and confusion matrix.

WHY A SEPARATE TEST SET?
    Using the val set for final evaluation would be over-optimistic: our training
    decisions (hyperparameters, early stopping) were driven by val performance.
    The test set provides an unbiased estimate of how the model will perform on
    NEW, unseen gesture trials — the goal of machine learning.

MODEL.PREDICT vs MODEL.EVALUATE
    model.predict(X_test)  : runs a forward pass; returns raw predictions (no loss).
    model.evaluate(X_test) : runs forward pass AND computes the loss + metrics.
    We use predict to get per-sample probabilities for the confusion matrix,
    and evaluate to get the single-number test accuracy / loss.
"""

# ── Run forward pass on test set ──────────────────────────────────────────────
# The model has two outputs: [class_probabilities, attention_weights]
pred_outputs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

# Handle both single-output and multi-output cases robustly
if isinstance(pred_outputs, (list, tuple)):
    y_prob    = pred_outputs[0]   # (N_test, N_classes) — one probability per class
    attn_test = pred_outputs[1]   # (N_test, T)          — attention weights
else:
    y_prob    = pred_outputs
    attn_test = None

# argmax: pick the class with the highest probability as the prediction
# e.g. [0.02, 0.05, 0.91, 0.02] → class 2 (Okay)
y_pred = np.argmax(y_prob, axis=1)

# ── Scalar metrics ────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(
    X_test,
    {'output': y_test_oh},
    batch_size = BATCH_SIZE,
    verbose    = 0,
)[:2]
print(f"Test accuracy : {test_acc:.4f}  |  Test loss : {test_loss:.4f}")

print("\n" + "─" * 60)
print("Classification Report:")
print("─" * 60)
# classification_report shows precision, recall, F1 for each class
# and macro/weighted averages.  COMP3308 week 6 metric definitions apply here.
report_str = classification_report(y_test, y_pred, target_names=LABELS, digits=4)
print(report_str)

# ── Confusion matrix ──────────────────────────────────────────────────────────
# Row i, Column j:  number of samples with true label i predicted as j.
# A perfect classifier has all mass on the diagonal (i == j).
cm      = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalise: per-class recall

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts — useful for seeing absolute numbers of errors
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, ax=axes[0],
)
axes[0].set_title('Confusion Matrix (counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted label')
axes[0].set_ylabel('True label')
axes[0].tick_params(axis='x', rotation=35)
axes[0].tick_params(axis='y', rotation=0)

# Normalised (row-normalised = per-class recall matrix)
# diagonal[i] = recall for class i
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, vmin=0, vmax=1, ax=axes[1],
)
axes[1].set_title('Confusion Matrix (row-normalised)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted label')
axes[1].set_ylabel('True label')
axes[1].tick_params(axis='x', rotation=35)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle(
    f'CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention — Test Set Evaluation',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── Per-class accuracy bar chart ──────────────────────────────────────────────
# Diagonal of the normalised CM = per-class recall = per-class accuracy
per_class_acc = cm_norm.diagonal()

# Colour-code bars: red < 70%, yellow 70-90%, green ≥ 90%
colours = [
    '#d73027' if v < 0.7 else ('#fee090' if v < 0.9 else '#1a9850')
    for v in per_class_acc
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(LABELS, per_class_acc * 100, color=colours, edgecolor='white')
ax.axhline(
    100 * per_class_acc.mean(), color='steelblue', linestyle='--',
    linewidth=1.5, label=f'Mean {100 * per_class_acc.mean():.1f}%',
)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=35)
ax.legend()
# Label each bar with its percentage value
for bar, v in zip(bars, per_class_acc):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
        f'{v * 100:.1f}%', ha='center', va='bottom', fontsize=9,
    )
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_per_class_accuracy.png'), dpi=120, bbox_inches='tight')
plt.show()


---
## Cell 9 — Attention Weight Visualisation

The attention layer assigns a weight $\alpha_t$ to every timestep $t$ in the gesture window.
Plotting these weights reveals **which part of the 5-second recording** was most discriminative
for each prediction:

- **High weight early in the sequence** → the gesture's *onset* carries the most information
  (e.g. the first sharp flex event is unique to this gesture).
- **High weight at the end** → the gesture's *release/hold phase* is most distinctive.
- **Spread weights** → the model uses the full trajectory rather than a single key moment.
- **Sharp peak mid-sequence** → there is a single diagnostic event (e.g. peak wrist rotation).

This is one of the key advantages of the attention mechanism over a plain LSTM:
a plain LSTM compresses everything into $h_T$ (the final state), making it impossible
to ask "which part of the sequence did you use?"  Attention makes this explicit and
auditable — an important property in a thesis and in safety-critical applications.

The second plot shows the **mean attention per class** across the entire test set.
Different gesture classes should focus on different temporal windows if the model
has learned genuinely class-specific temporal patterns.


In [ ]:
"""
Cell 9 — Attention weight extraction and visualisation.

WHAT WE ARE VISUALISING
────────────────────────
Recall from the BahdanauAttention layer:
    alpha_t = softmax(V · tanh(W · h_t))

alpha is a vector of T weights (one per BiLSTM timestep) that sum to 1.
High alpha_t means the model assigned high importance to the gesture signal
around time t when making its prediction for this sample.

KEY IMPLEMENTATION NOTE — TIMESTEP MISMATCH
    The attention weights have T_attn = T_cnn_out timesteps (e.g. 39 with two
    MaxPool(2) blocks applied to 156-timestep input).
    The original sensor signal has TARGET_LEN timesteps (e.g. 156).
    We linearly upsample the weights back to TARGET_LEN for overlay on the signal.
    The interpolation is a display convenience — no information is added or removed.

WHAT TO LOOK FOR
    - Do different gesture classes show different attention peaks?
    - Does the attention peak correspond to the most distinctive phase of the gesture?
    - Are incorrectly classified samples (✗) attending to less informative regions?
"""
from scipy.interpolate import interp1d as sci_interp


def get_attention_weights(model, X_batch):
    """
    Run a forward pass through the model and extract both outputs.

    Args:
        model   : the compiled Keras model (has two outputs: probs, attn_weights)
        X_batch : numpy array of shape (N, T, C)

    Returns:
        y_prob  : (N, C)  softmax class probabilities
        attn_w  : (N, T_attn)  per-timestep attention weights
                  (T_attn ≤ T because CNN pooling reduced the time dimension)

    Works for both 'additive' and 'scaled_dot' attention types because both
    expose attn_weights as the second model output.
    """
    outputs = model.predict(X_batch, verbose=0)
    if isinstance(outputs, (list, tuple)):
        return outputs[0], outputs[1]   # (probabilities, attention_weights)
    return outputs, None                 # fallback if model only has one output


def upsample_weights(w, target_len):
    """
    Linearly interpolate a 1-D attention weight vector from its native length
    (T_attn = T_original // pool_factor^n_blocks) back to target_len.

    Args:
        w          : 1-D numpy array of attention weights, shape (T_attn,)
        target_len : desired output length (= TARGET_LEN, the original signal length)

    Returns:
        Interpolated 1-D array of shape (target_len,).

    Note: this is a DISPLAY operation only.  The weights themselves are not changed
    conceptually — we are just spreading them back across the original time grid
    so they can be overlaid on the raw sensor signal.
    """
    if len(w) == target_len:
        return w   # already the right length — no interpolation needed
    # Create old and new normalised coordinate grids
    x_old = np.linspace(0, 1, len(w))
    x_new = np.linspace(0, 1, target_len)
    # sci_interp builds a linear interpolation function fitted to (x_old, w)
    return sci_interp(x_old, w, kind='linear')(x_new)


# ── Select one test sample per class for the overlay plot ────────────────────
# Preference: correctly classified samples (more representative of what the model
# has learned); fall back to any sample of the class if none are correct.
n_viz       = min(N_CLASSES, 6)   # visualise at most 6 classes
sample_idxs = []
for ci in range(n_viz):
    candidates = np.where(y_test == ci)[0]
    if len(candidates) == 0:
        continue
    correct = candidates[y_pred[candidates] == ci]   # correctly predicted indices
    idx     = correct[0] if len(correct) > 0 else candidates[0]
    sample_idxs.append(idx)

# Run forward pass to get probabilities and attention weights for these samples
X_viz             = X_test[sample_idxs]
y_true_viz        = y_test[sample_idxs]
probs_viz, attn_viz = get_attention_weights(model, X_viz)
y_pred_viz        = np.argmax(probs_viz, axis=1)

# ── Overlay plot: sensor signals + attention curve ────────────────────────────
# Each subplot shows:
#   BLUE lines  : the first 6 normalised sensor channels (thin, transparent)
#   RED region  : the upsampled attention weight (right y-axis)
#
# A spike in the red region marks the timestep where the model was "looking"
# when it made its prediction.
fig, axes = plt.subplots(n_viz, 1, figsize=(14, 3.2 * n_viz), sharex=False)
axes = np.array(axes).flatten()

# Time axis in seconds: 0 → TARGET_LEN / FS_HZ (e.g. 0 → 5.0 s)
time_axis = np.linspace(0, TARGET_LEN / FS_HZ, TARGET_LEN)

for i, (ax, idx) in enumerate(zip(axes, sample_idxs)):
    signal   = X_test[idx]              # (T, C) — normalised sensor values
    true_lbl = LABELS[y_true_viz[i]]
    pred_lbl = LABELS[y_pred_viz[i]]
    correct  = '✓' if y_true_viz[i] == y_pred_viz[i] else '✗'

    # Plot first 6 sensor channels as thin, semi-transparent blue lines
    # (showing all channels would be too cluttered)
    for ch in range(min(signal.shape[1], 6)):
        ax.plot(time_axis, signal[:, ch], alpha=0.35, linewidth=0.8, color='steelblue')

    if attn_viz is not None:
        # Upsample attention weights to original signal length for overlay
        raw_w  = attn_viz[i]                         # (T_attn,) — native attention length
        w_full = upsample_weights(raw_w, TARGET_LEN) # (T,)     — upsampled to signal length

        # Second y-axis (twinx) for the attention weights — avoids scale clash with sensor values
        ax2 = ax.twinx()
        ax2.fill_between(time_axis, w_full, alpha=0.3, color='tomato', label='Attention α')
        ax2.plot(time_axis, w_full, color='tomato', linewidth=1.2, alpha=0.7)
        ax2.set_ylabel('Attn weight α', color='tomato', fontsize=8)
        ax2.tick_params(axis='y', labelcolor='tomato', labelsize=7)
        # Set y-limit slightly above max weight so the fill_between is clearly visible
        ax2.set_ylim(0, max(w_full.max() * 1.4, 1e-6))

    ax.set_title(
        f'{correct} True: {true_lbl}  |  Predicted: {pred_lbl}  '
        f'(confidence: {probs_viz[i, y_pred_viz[i]]:.2f})',
        fontsize=10,
        color='darkgreen' if correct == '✓' else 'firebrick',
    )
    ax.set_ylabel('Norm. sensor value')
    ax.set_xlabel('Time (s)')
    ax.grid(True, alpha=0.2)

plt.suptitle(
    f'Attention Weights Overlaid on Sensor Signals  ({ATTENTION_TYPE} attention)\n'
    'Red shaded region = timesteps most influential for the model\'s decision',
    fontsize=12, fontweight='bold', y=1.005,
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_attention_visualisation.png'), dpi=120, bbox_inches='tight')
plt.show()


# ── Mean attention heatmap across all test samples ────────────────────────────
# For each class, average the attention weights over all test samples of that class.
# This reveals the typical temporal focus pattern for each gesture.
if attn_test is not None:
    print("\nAttention weight heatmap (mean per class):")

    T_attn     = attn_test.shape[1]                    # attention resolution
    mean_attn  = np.zeros((N_CLASSES, T_attn))         # (C, T_attn)

    for ci in range(N_CLASSES):
        ci_idx = np.where(y_test == ci)[0]             # test indices for class ci
        if len(ci_idx) > 0:
            mean_attn[ci] = attn_test[ci_idx].mean(axis=0)  # mean over samples

    # Upsample each class's mean attention to TARGET_LEN for a consistent x-axis
    mean_attn_full = np.stack(
        [upsample_weights(mean_attn[ci], TARGET_LEN) for ci in range(N_CLASSES)],
        axis=0,
    )  # (N_CLASSES, TARGET_LEN)

    fig, ax = plt.subplots(figsize=(14, max(3, N_CLASSES * 0.8)))
    sns.heatmap(
        mean_attn_full,
        ax         = ax,
        cmap       = 'YlOrRd',   # yellow (low) → orange → red (high)
        yticklabels= LABELS,
        cbar_kws   = {'label': 'Mean attention weight α'},
    )
    # Override x-ticks with time labels in seconds
    tick_positions = np.arange(0, TARGET_LEN, 10)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(
        [f"{v:.1f}s" for v in np.linspace(0, TARGET_LEN / FS_HZ, len(tick_positions))],
        rotation=45, fontsize=8,
    )
    ax.set_title(
        f'Mean Attention Weights per Class (test set, {ATTENTION_TYPE} attention)',
        fontsize=12, fontweight='bold',
    )
    ax.set_xlabel('Time into trial (s)')
    ax.set_ylabel('Gesture class')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '04_attention_heatmap.png'), dpi=120, bbox_inches='tight')
    plt.show()


---
## Cell 10 — Save Model and Results

Saves two artefacts:
1. **Keras model file** (`.keras`) — contains the full architecture, trained weights,
   and optimiser state.  Can be loaded with `tf.keras.models.load_model()`.
2. **TF SavedModel** directory — required for conversion to TensorFlow Lite (TFLite),
   which enables deployment on the ESP32-S3 microcontroller in the thesis hardware.
3. **Results JSON** — a structured record of all hyperparameters and evaluation metrics,
   used by `00_Results_Comparison.ipynb` to compare models side-by-side.


In [ ]:
"""
Cell 10 — Save the trained model and a structured results file.

WHY TWO MODEL FORMATS?
    .keras format     : Keras-native; includes full model graph, weights, compile state.
                        Best format for reloading in Python for further training or analysis.
    TF SavedModel     : TensorFlow's portable format; required input for tflite_convert
                        to produce the .tflite model deployed on the ESP32-S3 glove device.

WHY A RESULTS JSON?
    The 00_Results_Comparison notebook loads results JSONs from ALL model notebooks and
    builds a comparison table.  Every notebook writes the same key names so the
    aggregation is straightforward.
"""
import datetime

# ── Save final (best-weights) model ──────────────────────────────────────────
# EarlyStopping with restore_best_weights=True means model.weights are already
# the best seen during training — we just save them here.
final_model_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attention_final.keras')
model.save(final_model_path)
print(f"Model saved       : {final_model_path}")

# TF SavedModel export for TFLite conversion pipeline (edge deployment)
savedmodel_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attention_savedmodel')
model.export(savedmodel_path)
print(f"SavedModel saved  : {savedmodel_path}")

# ── Build evaluation summary dict ─────────────────────────────────────────────
report_dict = classification_report(
    y_test, y_pred, target_names=LABELS, output_dict=True
)

# ── Results JSON ──────────────────────────────────────────────────────────────
results = {
    "notebook"    : "04_CNN_BiLSTM_Attention",
    "timestamp"   : datetime.datetime.now().isoformat(),

    "architecture": {
        "type"            : "CNN + BiLSTM + Attention",
        "attention_type"  : ATTENTION_TYPE,
        "cnn_filters"     : CNN_FILTERS,
        "cnn_kernel_size" : CNN_KERNEL_SIZE,
        "cnn_pool_size"   : CNN_POOL_SIZE,
        "bilstm_units"    : BILSTM_UNITS,
        "dense_units"     : DENSE_UNITS,
        "total_params"    : int(model.count_params()),
    },

    "dataset"     : {
        "n_samples"    : int(X.shape[0]),
        "n_timesteps"  : int(TARGET_LEN),
        "n_features"   : int(N_FEATURES),
        "n_classes"    : int(N_CLASSES),
        "classes"      : LABELS,
        "filter_type"  : FILTER_TYPE,
        "normalization": NORMALIZATION,
        "fs_hz"        : FS_HZ,
    },

    "training"    : {
        "epochs_run"    : len(history.history[loss_key]),
        "epochs_config" : EPOCHS,
        "batch_size"    : BATCH_SIZE,
        "learning_rate" : LEARNING_RATE,
        "early_stop"    : EARLY_STOP_PATIENCE,
        "best_val_acc"  : float(max(history.history[val_acc_key])),
        "best_epoch"    : int(np.argmax(history.history[val_acc_key]) + 1),
    },

    "evaluation"  : {
        "test_accuracy" : float(test_acc),
        "test_loss"     : float(test_loss),
        "per_class"     : {
            lbl: {
                "precision": float(report_dict[lbl]["precision"]),
                "recall"   : float(report_dict[lbl]["recall"]),
                "f1-score" : float(report_dict[lbl]["f1-score"]),
                "support"  : int(report_dict[lbl]["support"]),
            }
            for lbl in LABELS if lbl in report_dict
        },
        "macro_avg"    : report_dict.get("macro avg",    {}),
        "weighted_avg" : report_dict.get("weighted avg", {}),
    },

    "artifacts"   : {
        "model_keras"       : final_model_path,
        "model_savedmodel"  : savedmodel_path,
        "class_distribution": os.path.join(RESULTS_DIR, '04_class_distribution.png'),
        "training_curves"   : os.path.join(RESULTS_DIR, '04_training_curves.png'),
        "confusion_matrix"  : os.path.join(RESULTS_DIR, '04_confusion_matrix.png'),
        "attention_viz"     : os.path.join(RESULTS_DIR, '04_attention_visualisation.png'),
        "attention_heatmap" : os.path.join(RESULTS_DIR, '04_attention_heatmap.png'),
    },
}

results_path = os.path.join(RESULTS_DIR, '04_cnn_bilstm_attention_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results JSON saved: {results_path}")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print(" CNN + BiLSTM + Attention — Final Results Summary")
print("═" * 60)
print(f"  Attention type        : {ATTENTION_TYPE}")
print(f"  Total parameters      : {model.count_params():,}")
print(f"  Training epochs ran   : {len(history.history[loss_key])} / {EPOCHS}")
print(f"  Best validation acc   : {max(history.history[val_acc_key]):.4f}")
print(f"  Test accuracy         : {test_acc:.4f}")
print(f"  Test loss             : {test_loss:.4f}")
print(f"  Macro F1              : {report_dict.get('macro avg', {}).get('f1-score', float('nan')):.4f}")
print("═" * 60)
print("\nPaper reference (A-CBLN, Wu et al. 2023):")
print("  Architecture   : 2D CNN → 2× BiLSTM(8) → Attention → Dense(16)")
print("  Reported acc   : 95.05%  on 7-class handwashing gestures")
print("═" * 60)
